### Install package


In [ ]:
# Install package
# !cd E:\Projects\SSA\AugusLabDP && pip install -e ".[utils]"

### Get all sessions statistics, and get one session

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Dict, Optional
import matplotlib.pyplot as plt

from AugusLabDP.utils.readout_utils import load_dataset, get_all_probe_mapping, get_anesthesia_period

data_folder = Path(r"C:\Users\bjmiao\The Augustine Lab Dropbox\Benjie Miao\Benjie_Jonny\SSA_Benjie\DPcachedata\\")
session_info_path = data_folder / 'session_info.csv'
df_session_info = pd.read_csv(session_info_path)
df_probe_mapping = get_all_probe_mapping(data_folder)
[print(f"{dataset}: {len(df_probe_mapping[dataset])} probes") for dataset in df_probe_mapping.keys()]
df_session_info.head() 

In [ ]:
session_index = 4  # '14T_5378529_AP_Amy_Day1_g0'
item = df_session_info.iloc[session_index]
print(item)
session_name = item['session']
print("Now loading session: ", session_name)
df = df_probe_mapping[item['dataset']]
df = df[df.session == item['session']]
probe_mapping = {probe:(probenum, depth) for probe, probenum, depth in zip(df['probe'], df['probenum'], df['probe_depth'])}
results = load_dataset(
    data_folder / item['dataset'], item['session'], item['type'],
    probe='all', probe_mapping = probe_mapping,
    need_modules=['spike', 'region', 'video', 'ttl', 'eeg', 'ecg', 'pupil', 'lfp'])
# ecg_r_peaks = find_r_peaks(results['ecg'], float(results['ttl_meta']['niSampRate']))

#### Load meta data, and experimental label tags

In [ ]:
results.keys()

In [ ]:
niSampRate = float(results['ttl_meta']['niSampRate']) # sampling rate of TTL, ECG and EEG
print(f"Sampling rate of TTL, ECG and EEG: {niSampRate} Hz")
print("Total recording time: ", results['ttl_camera'].shape[0] / niSampRate, " seconds")
print(f"Session start/stop/duration: {results['session_start_time']:.1f}s / {results['session_stop_time']:.1f}s / {results['session_duration']:.1f}s")

# Get experimental label tags
print("Experimental label tags:")
experimental_label_tag = results['experimental_label_tag']
for label, (start, stop) in experimental_label_tag.items():
    print("\t", label, start, stop)

start_anesthesia_period, stop_anesthesia_period = get_anesthesia_period(experimental_label_tag, before_seconds = 120, after_seconds = 600)
print(f"Anesthesia period: {start_anesthesia_period:.1f}s - {stop_anesthesia_period:.1f}s")

#### Load spiking data

In [ ]:
spike_times, spike_clusters = results['spike_times']
assert spike_times.shape == spike_clusters.shape
print("Number of spikes: ", spike_times.shape[0])

spike_matrix = results['spike_matrix']
print("Spike matrix shape: ", spike_matrix.shape)

### Load ECG data

In [ ]:
from AugusLabDP.utils.ecg_utils import find_r_peaks, get_heart_rate, ecg_to_bpm, plot_ecg_with_r_peaks
ecg = results['ecg']
niSampRate = float(results['ttl_meta']['niSampRate'])
r_peaks_in_seconds, threshold = find_r_peaks(ecg, niSampRate)
fig, ax = plot_ecg_with_r_peaks(ecg, r_peaks_in_seconds, start_time = 100, stop_time = 120, sampling_rate = niSampRate)
plt.show()
plt.close()

bpm = ecg_to_bpm(ecg, niSampRate, timebin = 1)
plt.plot(bpm)
plt.title("BPM")
plt.show()



### Load brain region

In [ ]:
from AugusLabDP.utils.brain_region_utils import plot_region_mark, get_meta_region_IBL, get_meta_region_by_target_list

meta_region_coarse = get_meta_region_by_target_list(results['cluster_region'])
print("Meta region (coarse): ", np.unique(meta_region_coarse))

meta_region_IBL = get_meta_region_IBL(results['cluster_region'])
print("Meta region (IBL): ", np.unique(meta_region_IBL))

plot_region_mark(results['cluster_region'], orientation = 'h', fill_ratio = 1)

### Load pupil size

In [ ]:
results.keys()

In [ ]:
from AugusLabDP.utils.pupil_utils import get_pupil_size
mean_pupil_size = get_pupil_size(results['pupil'])
plt.plot(mean_pupil_size)
plt.title("Pupil size")
plt.show()


### Load EEG/LFP


In [ ]:

from AugusLabDP.utils.eeg_utils import preprocess_group, multitaper_spectrogram, plot_spectrogram
print("Lfp shape: ", results['lfp'].shape)
print("Channel region: ", results['channel_region'].shape)


In [ ]:
SPEC_PARAMS = dict(
    window_s=2.0,
    step_s=0.5,
    bandwidth=4.0,
    fmin=0.5,
    fmax=200.0,
    adaptive=True,
    low_bias=True,
    n_jobs=1,
)

channel_regions, counts = np.unique(results['channel_region'], return_counts = True)
    # Get the region with the maximum channel count
example_region = channel_regions[np.argmax(counts)]

chans = np.where(results['channel_region'] == example_region)[0]
data_uV = results['lfp'][chans, :]
fs = results['lfp_fs']
x = preprocess_group(data_uV, fs, detrend=True, notch=60.0)
freqs, t_centers, S_db = multitaper_spectrogram(x, fs, **SPEC_PARAMS)

fig, (ax_sig, ax_spec) = plt.subplots(
    2, 1, figsize=(12, 6), gridspec_kw={"height_ratios": [1, 3]}
)
ax_sig.plot(x, lw=0.6, color="k")
ax_sig.set_ylabel("Mean LFP (\u00b5V)")
ax_sig.set_title(
    f"{example_region}: {len(chans)} channels"
)
ax_sig.grid(alpha=0.3)

plot_spectrogram(freqs, t_centers, S_db, ax=ax_spec)

fig.tight_layout()
plt.show()
plt.close(fig)

### Run all sessions function

This is a function that is easy to run analysis on all sessions

In [ ]:
from AugusLabDP.utils.run_in_all_sessions import run_in_all_sessions

# example function
def get_firing_rate(results, session_name, session_type, dataset, row):
    mean_firing_rate = results['spike_matrix'].mean() * 10 # 100ms
    print(f'Mean firing rate for session: {session_name}: {mean_firing_rate}')

run_in_all_sessions(get_firing_rate, data_folder)
